In [1]:
%run 00_config.py

In [2]:
from pymilvus import MilvusClient, DataType, Function, FunctionType
schema = MilvusClient.create_schema()
from pymilvus import CollectionSchema, FieldSchema, DataType

In [25]:
schema = CollectionSchema(fields=[
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="product_id", dtype=DataType.VARCHAR, max_length=32),
    FieldSchema(name="comprehensive_description", dtype=DataType.VARCHAR, max_length=3000, enable_analyzer=True),
    FieldSchema(name="sparse", dtype=DataType.SPARSE_FLOAT_VECTOR),
    ]
)

In [ ]:
bm25_function = Function(
    name="text_bm25_emb", # Function name
    input_field_names=["comprehensive_description"], # Name of the VARCHAR field containing raw text data
    output_field_names=["sparse"], # Name of the SPARSE_FLOAT_VECTOR field reserved to store generated embeddings
    function_type=FunctionType.BM25, # Set to `BM25`
)

schema.add_function(bm25_function)

In [32]:
import json
#do this for the very first time.
# am_products= load_dataset("bprateek/amazon_product_description")
# am_products = am_products['train']
am_products = []
with open('product-train.jsonl') as f:
    for line in f:
        am_products.append(json.loads(line))

In [27]:
client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus"
)

In [28]:
client.drop_collection(collection_name=FTS_collection_name)

In [ ]:
client.create_collection(
    collection_name = FTS_collection_name,
    schema=schema,
    database_name="default")
coll_metadata = [{
    "collection_name": FTS_collection_name,
    "primary_field_name": "id",
    "schema": -1,
    "index_params": [
        {
            "index_name": "sparse_index",
            "field_name": "sparse",
            "index_type": "SPARSE_INVERTED_INDEX",
            "metric_type": "BM25"
        },
        {
            "index_name": "product_id_index",
            "field_name": "product_id",
            "index_type": "INVERTED",
        }, 
]
    },
]
#Create each index with its own index_params (BM25 metric only valid on BM25 function output)
index_params = MilvusClient.prepare_index_params()
for index_param in coll_metadata[0]["index_params"]:
    index_params.add_index(index_name=index_param["index_name"], 
                            field_name=index_param["field_name"], 
                            index_type=index_param["index_type"], 
                            metric_type=index_param.get("metric_type", None))
client.create_index(
    collection_name=coll_metadata[0]["collection_name"],
    index_params=index_params,
    database_name="default"
)
client.flush(
    collection_name = FTS_collection_name,
    database_name="default"
)

In [30]:
import time
def get_entity( product_id: str, comprehensive_description: str) -> dict:

    # Prepare the data for insertion
    data = {
        #"id": None,  # Auto-generated ID
        "product_id": product_id,
        "comprehensive_description": comprehensive_description,
        #"sparse": None, #will become from BM25 function

    }
    #print(f"Processing product_id: {product_id}")

    return data

In [ ]:
for i in range(21):  # Adjust the range as needed
    print(f"Processing batch {i}")
    FTS_args = []
    for product in am_products[i*500: (i+1)*500]:  # Adjust the range as needed
        # get unique product ID from the product
        product_id = product.get('Uniq Id', '')
        
        # build comprehensive description
        comprehensive_description_array = []
        if (product.get('Product Name')) is not None:
            comprehensive_description_array.append(product.get('Product Name', ''))
        if (product.get('Category')) is not None:
            comprehensive_description_array.append(" categorized within " + ",".join(product.get('Category', '').split(' | ')))
        if (product.get('About Product')) is not None:
            comprehensive_description_array.append("with details - " + product.get('About Product', ''))
        comprehensive_description = ' '.join(comprehensive_description_array)

        FTS_args.append(( product_id, comprehensive_description))

    results = [get_entity(*arg_tuple) for arg_tuple in FTS_args]
    
    client.insert(
        collection_name=FTS_collection_name,
        data=results
    )


In [38]:
client.load_collection(
    collection_name=FTS_collection_name,
    database_name="default"
)

In [ ]:
client.flush(
    collection_name=FTS_collection_name,
    database_name="default"
)

In [ ]:
client.get_collection_stats(
    collection_name=FTS_collection_name,
)

In [41]:
search_params = {
    'params': {'drop_ratio_search': 0.2},
}
def run_text_search1(search_text: str):
    # Perform a text search using the BM25 function
    results_fts  = client.search(
        collection_name=FTS_collection_name, 
        data=[search_text],
        anns_field='sparse',
        output_fields=['comprehensive_description'], # Fields to return in search results; sparse field cannot be output
        limit=12,
        search_params=search_params
    )
    return results_fts

In [ ]:
# GET DATA
import ipywidgets as widgets
from ipywidgets import interact_manual

results_display = widgets.HTML("<h3>Results:</h3>")
def run_text_search(search: str):
    r = run_text_search1(search)
    rows = [f"<tr><td>{i.distance}</td><td border=1>{i.entity['comprehensive_description']}</td></tr>" for i in r[0] ]
    results_display.value = f"<table width='1200'><tr><th><b>Product ID</b></th><th><b>Description</b></th></tr>{''.join(rows)}</table>"
    


# DISPLAY BUTTON

interact_manual(run_text_search, search="Doctor play set for kids with accessories");
display(results_display)